# Exploratory data analysis — Breast Cancer (tabular task)

Generated by `scripts/build_eda_notebook.py`. It uses the same code path as
`dltf train --config configs/tabular_mlp.yaml`, so what you see here is what the
model sees: a median imputer + standard scaler fitted **only** on the training
split.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dltf.data.splits import stratified_split
from dltf.data.tabular import build_eda, load_tabular_frame
from dltf.seed import set_seed

set_seed(42)
frame = load_tabular_frame("breast_cancer")
df = frame.as_frame()
print(f"{df.shape[0]} rows, {df.shape[1] - 1} features, classes: {frame.class_names}")
df.head()

## Splits and class balance

Stratified 70/15/15 split, identical to training (seed 42). Every class keeps its
proportion in each split.

In [ ]:
split = stratified_split(frame.y, val_fraction=0.15, test_fraction=0.15, seed=42)
balance = pd.DataFrame(
    {
        name: np.bincount(frame.y[idx], minlength=frame.num_classes)
        for name, idx in split.items()
    },
    index=frame.class_names,
)
print("split sizes:", {k: int(v.size) for k, v in split.items()})
balance

## Missing values and correlations

The EDA helper writes a JSON summary plus PNG figures into `artifacts/eda/`.

In [ ]:
summary = build_eda(frame, "artifacts/eda")
print("missing values:", summary["n_missing_total"])
print("duplicate rows:", summary["duplicate_rows"])
print("class balance:", summary["class_balance"])

top = pd.Series(summary["top_correlated_with_target"]).sort_values(key=np.abs, ascending=False)
top.head(8)

## Leakage check

Standardisation statistics are fitted on the training rows only. If we accidentally
fitted on the whole dataset the two mean vectors below would be identical.

In [ ]:
from sklearn.preprocessing import StandardScaler

train = split["train"]
scaler = StandardScaler().fit(frame.X[train])
print("train-only mean (first 5):", np.round(scaler.mean_[:5], 4))
print("full-data mean (first 5):", np.round(frame.X.mean(axis=0)[:5], 4))
assert not np.allclose(scaler.mean_, frame.X.mean(axis=0)), "leakage detected"
print("no leakage: train-only statistics differ from full-data statistics")

plt.close("all")